# ML Work

## Predicting Ages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('../data/titanic_eda.csv')
df

Importing issential libraries for ML

In [ ]:

from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV      
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso


In [ ]:
df_train = df.copy()

In [ ]:
df_train['Embarked_S'] = df_train['Embarked'].apply(lambda x: 1 if x == 'S' else 0)
df_train['Embarked_C'] = df_train['Embarked'].apply(lambda x: 1 if x == 'C' else 0)
df_train['Embarked_Q'] = df_train['Embarked'].apply(lambda x: 1 if x == 'Q' else 0)
df_train['lone_traveler'] = df_train['lone_traveler'].apply(lambda x: 1 if x == 'Yes' else 0)
df_train['Sex'] = df_train['Sex'].replace({'male': 0, 'female': 1})

df_train = df_train.drop(columns=['Name', 'Ticket', 'surname', 'Embarked'])


df_train

In [ ]:
#X = df_train.drop(columns=['Age'])
#y = df_train['Age']

with_age = df_train[df_train['Age'].notna()]
without_age = df_train[df_train['Age'].isna()]
feature_cols = [c for c in df_train.columns if c not in ['Age', 'Survived']]
X = with_age[feature_cols]
y = with_age['Age']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42    
)


In [ ]:
param_grid = {
    'model__alpha': [0.001, 0.01, 0.1,1.0]}

In [ ]:
pipe1 = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),   
    ('model' ,  Ridge(alpha=1.0))    
])

In [ ]:
grid = GridSearchCV(pipe1, param_grid, cv=5)
grid.fit(X_train, y_train)


In [ ]:
best_alpha = grid.best_params_['model__alpha']
print(f"Best alpha: {best_alpha}")

In [ ]:
y_pred_lr = grid.predict(X_test)   

In [ ]:
mae = mean_absolute_error(y_test,y_pred_lr )
mae

In [ ]:
r2 = r2_score(y_test,y_pred_lr)
r2

In [ ]:
rmse = root_mean_squared_error(y_test,y_pred_lr)
rmse

In [ ]:
train_score = grid.score(X_train, y_train)
test_score = grid.score(X_test, y_test)

print("Train:", train_score)
print("Test:", test_score)

In [ ]:
pipe2 = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),   
    ('sgdr' ,SGDRegressor(
        penalty='elasticnet', max_iter=1000,l1_ratio=0.5
    ))    
])

In [ ]:
param_grid = {
    'sgdr__alpha': [0.0001, 0.001, 0.01],
    'sgdr__l1_ratio': [0.2, 0.5, 0.8],
    'sgdr__learning_rate': ['constant', 'optimal', 'invscaling', 'adaptive']
}

In [ ]:
grid_sgd = GridSearchCV(pipe2, param_grid, cv=5)
grid_sgd.fit(X_train , y_train )

In [ ]:
y_pred_sgdr = grid_sgd.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test,y_pred_sgdr )
mae

In [ ]:
r2 = r2_score(y_test,y_pred_sgdr)
r2

In [ ]:
rmse = root_mean_squared_error(y_test,y_pred_sgdr)
rmse

In [ ]:
print("Train:", grid_sgd.score(X_train, y_train))
print("Test:", grid_sgd.score(X_test, y_test))

In [ ]:
results = {
    "Ridge": {
        "MAE" : mean_absolute_error(y_test, y_pred_lr),
        "RMSE": root_mean_squared_error(y_test, y_pred_lr),
        "R2"  : r2_score(y_test, y_pred_lr),
        "Train R2": grid.score(X_train, y_train),
        "Test R2" : grid.score(X_test,  y_test),
    },
    "SGDRegressor": {
        "MAE" : mean_absolute_error(y_test, y_pred_sgdr),
        "RMSE": root_mean_squared_error(y_test, y_pred_sgdr),
        "R2"  : r2_score(y_test, y_pred_sgdr),
        "Train R2": grid_sgd.score(X_train, y_train),
        "Test R2" : grid_sgd.score(X_test,  y_test),
    },
}

results_df = pd.DataFrame(results).T.round(4)

results_df

In [ ]:
best_model_name =results_df['R2'].idxmax()
best_model = grid if best_model_name == "Ridge" else grid_sgd
print(f"Best model: {best_model_name}")

In [ ]:
X_missing = without_age[feature_cols]
print("X_missing rows  :", len(X_missing))     

predicted_ages = best_model.predict(X_missing)
print("Predictions made:", len(predicted_ages))


- As we can see here our R2 score is not that good cause we dont have enough features to predicit AGE but this pridictions are better than using mean 


In [ ]:
without_age['Age'] = predicted_ages

In [ ]:
df['Age'] = df_train['Age'].fillna(without_age['Age'])   

In [ ]:
df